# Véletlenszerű játszma Stockfish-elemzése + LLM narráció API hívással

**Pipeline:**
1. Véletlenszerű játszma sorsolása a `mychessdotcomgames.parquet` fájlból
2. Stockfish elemzés (centipawn annotációk lépésenként)
3. Annotált PGN exportálása → `output/llm-analysis/elemzett.pgn`
4. LLM API: didaktikus narráció generálása (provider: `config.LLM_PROVIDER`)
5. Narráció mentése → `output/llm-analysis/szoveges/{llm_provider}_{game_id}.txt`
6. TTS hangfájl generálása → `output/llm-analysis/hangos_narracio/{llm_provider}_{game_id}.mp3`

In [17]:
import os
import sys
import random
import shutil

# ROOT_DIR: a könyvtár ahol a config.py van
_cwd = os.getcwd()
ROOT_DIR = _cwd if os.path.exists(os.path.join(_cwd, "config.py"))            else os.path.abspath(os.path.join(_cwd, ".."))
sys.path.insert(0, ROOT_DIR)

import polars as pl
import chess
import chess.pgn
import chess.engine

import config
from src.llm_client import generate_text

print(f"ROOT_DIR    : {ROOT_DIR}")
print(f"LLM provider: {config.LLM_PROVIDER}  ({config.LLM_MODEL})")
print(f"API kulcs   : {chr(39)+chr(79)+chr(75)+chr(39) if config.LLM_API_KEY else chr(39)+'HIÁNYZIK – töltsd ki a secrets.py-t!'+chr(39)}")

ROOT_DIR    : D:\Workspace\chess-pgn-analysis
LLM provider: openai  (gpt-4o-mini)
API kulcs   : 'OK'


## 1. Véletlen játszma kiválasztása

In [18]:
PARQUET_PATH = os.path.join(ROOT_DIR, "output", "parquet", "mychessdotcomgames.parquet")

df = pl.read_parquet(PARQUET_PATH)
print(f"Összes játszma a parquet-ban: {len(df):,}")

idx = random.randint(0, len(df) - 1)
row = df.row(idx, named=True)

print(f"\nKiválasztott játszma (sor-index={idx}, game_id={row['game_id']}):")
print(f"  Fehér  : {row['white']} ({row['white_elo']})")
print(f"  Fekete : {row['black']} ({row['black_elo']})")
print(f"  Megnyitó: {row['eco']} · {row['opening']}")
print(f"  Eredmény: {row['result']}  |  Lépések: {row['num_moves']}")

Összes játszma a parquet-ban: 1,377

Kiválasztott játszma (sor-index=1344, game_id=1345):
  Fehér  : MyUsername_Already_Taken (1032)
  Fekete : Wujajin (1051)
  Megnyitó: B10 · 
  Eredmény: 0-1  |  Lépések: 138


## 2. Stockfish elemzés: heurisztika az LLM-nek!

A kisebb depth gyorsabb futást eredményez, de a magasabb alaposabb (pontosabb) elemzést!

In [19]:
def find_stockfish() -> str:
    if getattr(config, 'STOCKFISH_PATH', None) and os.path.isfile(config.STOCKFISH_PATH):
        return config.STOCKFISH_PATH
    bundled = os.path.join(ROOT_DIR, "stockfish", "stockfish-windows-x86-64-avx2.exe")
    if os.path.isfile(bundled):
        return bundled
    found = shutil.which("stockfish")
    if found:
        return found
    raise FileNotFoundError("Stockfish nem található! Ellenőrizd a stockfish/ mappát.")

SF_PATH = find_stockfish()
print(f"Stockfish: {SF_PATH}")

Stockfish: D:\Workspace\chess-pgn-analysis\stockfish\stockfish-windows-x86-64-avx2.exe


In [20]:
import asyncio
from tqdm.notebook import tqdm

if hasattr(asyncio, 'WindowsProactorEventLoopPolicy'):
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

DEPTH       = 12
MOVES_LIMIT = config.STOCKFISH_MOVES_LIMIT

moves_uci_list = row['moves_uci'].strip().split()
to_analyze     = min(len(moves_uci_list), MOVES_LIMIT)
print(f"Elemzés: mélység={DEPTH}, lépések={to_analyze}/{len(moves_uci_list)}\n")

board       = chess.Board()
evaluations = []

with chess.engine.SimpleEngine.popen_uci(SF_PATH) as engine:
    limit = chess.engine.Limit(depth=DEPTH)

    for i, uci in enumerate(tqdm(moves_uci_list[:MOVES_LIMIT], desc="Stockfish", unit="lépés")):
        move = chess.Move.from_uci(uci)
        if move not in board.legal_moves:
            print(f"  Illegális lépés: {uci} – leállítás")
            break

        san = board.san(move)
        board.push(move)

        info  = engine.analyse(board, limit)
        score = info["score"].white()

        evaluations.append({
            "move_number": (i // 2) + 1,
            "color":       "white" if i % 2 == 0 else "black",
            "uci":         uci,
            "san":         san,
            "cp":          score.score(mate_score=10000),
            "mate":        score.mate(),
            "fen":         board.fen(),   # állás a lépés UTÁN – Streamlit-anchor
        })

print(f"\nKész: {len(evaluations)} lépés elemezve.")


Elemzés: mélység=12, lépések=40/138



Stockfish:   0%|          | 0/40 [00:00<?, ?lépés/s]


Kész: 40 lépés elemezve.


## 3. Annotált PGN exportálása

In [21]:
game_id   = row["game_id"]
os.makedirs(config.LLM_ANALYSIS_DIR, exist_ok=True)
PGN_PATH  = config.ELEMZETT_PGN

# Duplikátum-ellenőrzés
pgn_exists     = os.path.exists(PGN_PATH) and os.path.getsize(PGN_PATH) > 0
already_in_pgn = False
if pgn_exists:
    with open(PGN_PATH, encoding="utf-8") as f:
        if f'[ParquetGameId "{game_id}"]' in f.read():
            already_in_pgn = True

if already_in_pgn:
    print(f"⚠️  Játszma #{game_id} már szerepel az elemzett.pgn-ben – kihagyva.")
else:
    game_pgn = chess.pgn.Game()
    game_pgn.headers.update({
        "Event":         row.get("event",    "?"),
        "Site":          row.get("site",     "?"),
        "Date":          row.get("date",     "?"),
        "White":         row.get("white",    "?"),
        "Black":         row.get("black",    "?"),
        "Result":        row.get("result",   "*"),
        "WhiteElo":      str(row.get("white_elo", "?")),
        "BlackElo":      str(row.get("black_elo", "?")),
        "ECO":           row.get("eco",      "?"),
        "Opening":       row.get("opening",  "?"),
        "ParquetGameId": str(game_id),
    })

    node = game_pgn
    for ev in evaluations:
        node = node.add_variation(chess.Move.from_uci(ev["uci"]))
        if ev["mate"] is not None:
            node.comment = f"[%eval #{ev['mate']}]"
        elif ev["cp"] is not None:
            node.comment = f"[%eval {ev['cp'] / 100:.2f}]"

    # StringExporter garantál egységes kimenetet; .strip()+"\n" pontosan egy \n-re zárja a játszmát.
    # Ha már van tartalom, egy "\n" elé kerül → mindig pontosan egy üres sor lesz a játszmák között.
    pgn_str = game_pgn.accept(chess.pgn.StringExporter(headers=True, variations=True, comments=True))
    with open(PGN_PATH, "a", encoding="utf-8") as f:
        if pgn_exists:
            f.write("\n")
        f.write(pgn_str.strip() + "\n")

    with open(PGN_PATH, encoding="utf-8") as f:
        content = f.read()
    game_count = content.count("[Event ")
    print(f"PGN hozzáfűzve: {PGN_PATH}")
    print(f"Összes elemzett játszma: {game_count}")
    print("\nUtolsó hozzáfűzött játszma (első 600 karakter):")
    last_start = content.rfind("[Event ")
    print(content[last_start:last_start + 600])

PGN hozzáfűzve: D:\Workspace\chess-pgn-analysis\output\llm-analysis\elemzett.pgn
Összes elemzett játszma: 5

Utolsó hozzáfűzött játszma (első 600 karakter):
[Event "Live Chess"]
[Site "Chess.com"]
[Date "2026.04.27"]
[Round "?"]
[White "MyUsername_Already_Taken"]
[Black "Wujajin"]
[Result "0-1"]
[WhiteElo "1032"]
[BlackElo "1051"]
[ECO "B10"]
[Opening ""]
[ParquetGameId "1345"]

1. e4 { [%eval 0.46] } 1... c6 { [%eval 0.76] } 2. Nf3 { [%eval 0.34] } 2... d5
{ [%eval 0.31] } 3. e5 { [%eval 0.38] } 3... Bg4 { [%eval 0.39] } 4. d4
{ [%eval 0.33] } 4... e6 { [%eval 0.31] } 5. h3 { [%eval -0.17] } 5... Bh5
{ [%eval 0.31] } 6. Bg5 { [%eval -3.68] } 6... Be7 { [%eval 0.37] } 7. Bxe7
{ [%eval -0.56] } 7... Nxe7 { [%eval -0.56] } 8. g4 { [%eval -0.52] } 8.


## 4. LLM narráció generálása

In [22]:
import json as _json

SYSTEM_PROMPT = """\
Te egy magyar sakkkommentátor vagy. Lépésenkénti Stockfish-elemzést kapsz JSON formátumban,
minden lépésnél a FEN-nel (az állás a lépés után) és a centipawn (cp) értékkel.

## Magyar sakkszakkifejezések – KIZÁRÓLAG ezeket használd:
- megnyitás: a játszma első ~10 lépése; cél a figurák fejlesztése, centrum ellenőrzése, sáncolás
- centrum: az e4, d4, e5, d5 mezők összessége – aki uralja, az irányítja a játékot
- fejlesztés: figurák kiindulási mezőről aktív pozícióba való mozgatása a megnyitásban
- sáncolás: a király biztonságba helyezése (rövid sánc: királyszárny; hosszú sánc: vezérszárny)
- taktikai motívum: rövid, konkrét lépéssorozaton alapuló előny (villa, nyársalás, kettős támadás, felfedett sakk)
- villa: egy figura egyszerre két ellenséges figurát támad
- nyársalás: értékes figurát megtámadva mögötte lévő gyengébbet nyeri meg
- stratégiai előny: hosszú távú pozicionális fölény (gyenge mező, tornyok nyílt vonalon, előretolt gyalog)
- gyenge mező: amelyet az ellenfél gyalogjai nem tudnak védeni – ideális beugrási pont
- nyílt vonal: gyalogoktól mentes oszlop, ahol a bástya különösen hatékony
- anyagi egyensúly: figurák értéke összehasonlítva (gyalog=1, huszár/futó=3, bástya=5, vezér=9)
- döntő lépés / fordulópont: ahol a cp érték legalább 150-et változik egy lépésen belül
- mattfenyegetés: közvetlen mattot kilátásba helyező lépés vagy lépéssorozat

## A Stockfish cp-értékek értelmezése (fehér nézőpontjából, pozitív = fehér előny):
- |cp| < 50:   kiegyenlített állás
- 50–150:      enyhe előny
- 150–300:     jelentős előny
- > 300:       döntő fölény
- mate != null: mattfenyegetés

## Kötelező lépésjelölés (TTS miatt – soha ne írj algebrai jelölést):
- Nxe5   → huszár üti e5-öt
- Rxh7   → bástya üti h7-et
- Bxf6   → futó üti f6-ot
- Qd8+   → vezér d8-ra, sakk
- O-O    → rövid sánc
- O-O-O  → hosszú sánc
- e4     → gyalog e4-re
- exd5   → gyalog üti d5-öt

## A narráció szerkezete – PONTOSAN 3 bekezdés:
1. Megnyitás és fejlesztés (1–10. lépés): nevezd meg az ECO-kód alapján a megnyitást,
   értékeld a figurák fejlesztését és a centrumharcot.
2. Középjáték és fordulópont: azonosítsd azt a lépést, ahol a cp a legnagyobb ugrást mutatja
   (legalább 150 cp); ezt nevezd fordulópontnak és magyarázd el miért volt döntő.
3. Összefoglalás és tanulság: mi volt a vesztes fél fő hibája? Milyen általános sakkelvvel függ össze?

## Kimenet – KIZÁRÓLAG valid JSON, se bevezető szöveg, se markdown kódblokk:
{
  "paragraphs": [
    {
      "text": "Bekezdés szövege, amelyben a lépésekre TTS-barát módon hivatkozol.",
      "anchors": [
        {"fen": "<FEN az adott lépés után>", "trigger_word": "<szó szerinti idézet a text-ből>"}
      ]
    }
  ]
}

Minden bekezdéshez annyi anchor kell, ahány konkrét lépésre hivatkozol.
A trigger_word pontosan azokat a szavakat tartalmazza, amelyek a text-ben szerepelnek (szó szerinti egyezés!).
"""

USER_PROMPT = (
    f"ECO: {row.get('eco', '?')} – {row.get('opening', '?')}\n"
    f"Fehér: {row.get('white', '?')} ({row.get('white_elo', '?')}), "
    f"Fekete: {row.get('black', '?')} ({row.get('black_elo', '?')})\n"
    f"Eredmény: {row.get('result', '?')}\n\n"
    "Elemzendő lépések (JSON):\n"
    + _json.dumps(evaluations, ensure_ascii=False, indent=2)
)

raw_response = generate_text(USER_PROMPT, system_prompt=SYSTEM_PROMPT)

# JSON tisztítás: markdown kódblokk eltávolítása, ha a modell mégis berakta
raw_clean = raw_response.strip()
if raw_clean.startswith("```"):
    raw_clean = raw_clean.split("```", 2)[1]
    if raw_clean.startswith("json"):
        raw_clean = raw_clean[4:]
    raw_clean = raw_clean.rsplit("```", 1)[0].strip()

try:
    narration_json = _json.loads(raw_clean)
    print("✅ JSON parse sikeres.")
    for i, para in enumerate(narration_json.get("paragraphs", []), 1):
        print(f"\n--- {i}. bekezdés ---")
        print(para["text"])
        print(f"  ({len(para.get('anchors', []))} anchor)")
except _json.JSONDecodeError as e:
    print(f"⚠️  JSON parse hiba: {e}")
    print("Nyers válasz:")
    print(raw_response)
    narration_json = None


✅ JSON parse sikeres.

--- 1. bekezdés ---
A játszma a B10 ECO kóddal kezdődött, ez a Caro-Kann védelem egy felfedezéssel. Fehér az első lépésben e4-gyel indított, amely után Fekete c6-ot lépett, ezzel biztosítva a d5-öt. Fehér a második lépésében Nf3-mal fejlesztette huszárát, míg Fekete d5-öt lépett, ami megnyitotta a centrumot. A harmadik lépés Fehér e5-ös lépése, amely stabil centrumot biztosít, azonban Fekete Bg4 lépésével próbálja megakadályozni a Fehér futó lépését. A negyedik lépésben Fehér d4-gyel próbálja fejleszteni a játékát, amire Fekete e6-ot lépett a kiszolgáltatott bástya védelmére. Az ötödik lépésnél Fehér h3-at lép, ami viszont tehetetlenséget mutat, és Fekete Bh5-tel válaszol, teret adva a gyalogos központ denevér-változatának. A hatodik lépés egy hiba, Fehér Bg5-t lép, amely hatalmas hátrányt okoz, mert a Fekete futó még mindig aktívan támad a d7 és d4 mezőkön.
  (10 anchor)

--- 2. bekezdés ---
A középjátékban a fordulópont a hatodik lépésben történt, amikor Fehér 

## 5. Narráció mentése

In [23]:
import json as _json

if narration_json is None:
    print("⚠️  Nincs valid JSON narráció – mentés kihagyva.")
else:
    llm_name = config.LLM_PROVIDER
    json_name = f"{llm_name}_{game_id}.json"
    txt_name  = f"{llm_name}_{game_id}.txt"

    os.makedirs(config.LLM_ANALYSIS_JSON_DIR, exist_ok=True)
    os.makedirs(config.LLM_ANALYSIS_SZOVEGES_DIR, exist_ok=True)

    JSON_PATH = os.path.join(config.LLM_ANALYSIS_JSON_DIR, json_name)
    LLM_PATH  = os.path.join(config.LLM_ANALYSIS_SZOVEGES_DIR, txt_name)

    # JSON fájl (FEN-anchorokat tartalmaz – Streamlit-nek)
    with open(JSON_PATH, "w", encoding="utf-8") as f:
        _json.dump(narration_json, f, ensure_ascii=False, indent=2)
    print(f"JSON narráció elmentve: {JSON_PATH}")

    # TTS szöveges fájl (csak a text mezők, anchor nélkül)
    tts_text = "\n\n".join(
        para["text"] for para in narration_json.get("paragraphs", [])
    )
    with open(LLM_PATH, "w", encoding="utf-8") as f:
        f.write(tts_text)
    print(f"TTS szöveg elmentve: {LLM_PATH}")
    print(f"\nTTS szöveg előnézete:")
    print(tts_text[:400], "..." if len(tts_text) > 400 else "")


JSON narráció elmentve: D:\Workspace\chess-pgn-analysis\output\llm-analysis\json_narracio\openai_1345.json
TTS szöveg elmentve: D:\Workspace\chess-pgn-analysis\output\llm-analysis\szoveges\openai_1345.txt

TTS szöveg előnézete:
A játszma a B10 ECO kóddal kezdődött, ez a Caro-Kann védelem egy felfedezéssel. Fehér az első lépésben e4-gyel indított, amely után Fekete c6-ot lépett, ezzel biztosítva a d5-öt. Fehér a második lépésében Nf3-mal fejlesztette huszárát, míg Fekete d5-öt lépett, ami megnyitotta a centrumot. A harmadik lépés Fehér e5-ös lépése, amely stabil centrumot biztosít, azonban Fekete Bg4 lépésével próbálja me ...


## 6. TTS hangfájl generálása (kicsit lassabban fut!)

In [24]:
from src.tts_client import generate_audio

mp3_name = txt_name.replace(".txt", ".mp3")
os.makedirs(config.LLM_ANALYSIS_HANGOS_DIR, exist_ok=True)
MP3_PATH = os.path.join(config.LLM_ANALYSIS_HANGOS_DIR, mp3_name)

if os.path.exists(MP3_PATH):
    print(f"⚠️  Hangfájl már létezik: {MP3_PATH} – kihagyva.")
else:
    with open(LLM_PATH, encoding="utf-8") as f:
        narration_text = f.read()
    generate_audio(narration_text, MP3_PATH, show_progress=True)
    print(f"Hangfájl elmentve: {MP3_PATH}  ({os.path.getsize(MP3_PATH):,} bájt)")

TTS letöltés (OpenAI): 0.00B [00:00, ?B/s]

Hangfájl elmentve: D:\Workspace\chess-pgn-analysis\output\llm-analysis\hangos_narracio\openai_1345.mp3  (2,230,560 bájt)


## Végső állás – sakktábla

Az utolsó FEN-anchor alapján.

In [ ]:
import chess
import chess.svg
from IPython.display import SVG, display

# Az utolsó anchor FEN-je a generált JSON-ból
_last_fen = None
if narration_json and narration_json.get("paragraphs"):
    for _para in reversed(narration_json["paragraphs"]):
        for _anchor in reversed(_para.get("anchors", [])):
            if _anchor.get("fen"):
                _last_fen = _anchor["fen"]
                break
        if _last_fen:
            break

if _last_fen:
    _board = chess.Board(_last_fen)
    _svg   = chess.svg.board(_board, size=440)
    display(SVG(_svg))
    print(f"FEN: {_last_fen}")
else:
    print("Nincs megjeleníthető FEN – futtasd le előbb a narráció-generáló cellát!")
